In [ ]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import pandas as pd
import polars as pl
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from sklearn.cluster import DBSCAN, HDBSCAN, OPTICS
from sklearn.neighbors import KDTree
from sklearn.neighbors import NearestNeighbors

import sys


from pyS3M import IOFunctions

IO = IOFunctions.IO_Functions()

from pyS3M import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from pyS3M import PlottingBase

plotter = PlottingBase.PublicationPlotter(dark_background=False)

from pyS3M import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from pyS3M import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from pyS3M import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from pyS3M import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs(camera="zwo")

from pyS3M import MaskFunctions

M_F = MaskFunctions.Mask_Functions(camera="zwo")

from pyS3M import SpotDetectionFunctions

SD_F = SpotDetectionFunctions.SpotDetection_Functions(camera="zwo")

from pyS3M import SR_Functions

SupRes_F = SR_Functions.SuperRes_Functions(camera="zwo")

from pyS3M import HelperFunctions

H_F = HelperFunctions.Helper_Functions()

from pyS3M import render
from src import postprocess as _postprocess
from pyS3M import FiducialDetection

FD = FiducialDetection.FiducialDetector()


import types

import pyS3M.DriftCorrectionFunctions as DCF
from pyS3M.LinkingFunctions import link_localisations


smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

In [ ]:
data_folder = "../../Camera_Calibrations/ZWO_Camera"
gain_map = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset_map = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
read_noise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))
R, G, B, wavelength = S_F.getpixelefficiency()

pixel_QYs = np.vstack([B, G, R])
camera_parameters = {}
camera_parameters["pixel_QYs"] = pixel_QYs
camera_parameters["pixel_order"] = ["B", "G", "R"]
camera_parameters["pixel_order_indices"] = [0, 1, 2]

In [ ]:
image_folder = "/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/Brendan/20260623_MASSIVECELLS/ATTO594_Cy3B/250pM_F1_ATTO594_750pM_F2_Cy3B_1"

In [ ]:
localisation_files = H_F.file_search(image_folder, ".h5", "")

In [ ]:
localisation_files

In [ ]:
tif_files = H_F.file_search(image_folder, ".tif", "")

In [ ]:
metadata_files = H_F.file_search(image_folder, "metadata", "")

In [ ]:
x_coord, y_coord, width, height = IO.metadata_reader_imageJ(metadata_files[0])
n_frames = IO.metadata_nframes_reader_imageJ(metadata_files[0])

In [ ]:
loc_data = pd.read_hdf(localisation_files[0])

In [ ]:
loc_data = loc_data[loc_data["xc_err"] < 40/72]
loc_data = loc_data[loc_data["yc_err"] < 40/72]

loc_data = loc_data[loc_data["xc_err"] > 0]
loc_data = loc_data[loc_data["yc_err"] > 0]

loc_data = loc_data[loc_data["s_x_err"] < 40/72]
loc_data = loc_data[loc_data["s_y_err"] < 40/72]

loc_data = loc_data[(loc_data["s_x"] > 100/72) & (loc_data["s_x"] < 200/72)]
loc_data = loc_data[(loc_data["s_y"] > 100/72) & (loc_data["s_y"] < 200/72)]

loc_data = loc_data[loc_data["A_B_err"] < 0.01]
loc_data = loc_data[loc_data["A_G_err"] < 0.01]
loc_data = loc_data[loc_data["A_R_err"] < 0.01]

loc_data = loc_data[loc_data["A_B"] > 0.005]
loc_data = loc_data[loc_data["A_G"] > 0.05]
loc_data = loc_data[loc_data["A_R"] > 0.5]

loc_data = loc_data[loc_data["bg_B_err"] < 0.15]
loc_data = loc_data[loc_data["bg_G_err"] < 0.15]
loc_data = loc_data[loc_data["bg_R_err"] < 0.15]

loc_data = loc_data[~((loc_data['photons'] < 500) & (loc_data['spot_matched_filter_response'] < 25))]

loc_data = loc_data[loc_data["spot_background_std"] > 4]

#loc_data = loc_data[loc_data["photons"] < 100000]
#loc_data = loc_data[loc_data["photons"] > 100]


loc_data = loc_data[loc_data["chi_sqr"] < 3]

In [ ]:
plt.hist(loc_data['A_R'], 1000, alpha=0.5, color='darkred');
plt.hist(loc_data['A_G'], 1000, alpha=0.5, color='darkgreen');
#plt.hist(loc_data['A_B'], 1000, alpha=0.5, color='lightblue');
plt.xlim([0, 1])
plt.show()

In [ ]:
image = render.render(
    locs=loc_data.to_records(index=False),
    oversampling=1,
    viewport=((0, 0), (height, width)),
    blur_method="smooth",
)[1]

In [ ]:
fig, axs = plotter.two_column_plot()

axs = plotter.image_plot(axs, image, colorbar=False)
plt.show()

In [ ]:
drift_corrector = DCF.Drift_Correction_Functions()

info = [{
    "Width": width,         # Image width in pixels
    "Height": height,        # Image height in pixels
    "Frames": np.max(loc_data['frame']),      # Total number of frames in the movie
    "Pixelsize": 72,        # pixel size always in nm
}]

corrected_locs, drift_result = drift_corrector.undrift(locs=loc_data.to_records(index=False),
                                          info=info,
                                          method="aim",
                                          segmentation=20,
                                            intersect_d=20/72,
                                          roi_r=60/72,
                                         )
corrected_locs = pd.DataFrame(corrected_locs)

In [ ]:
plt.plot(drift_result.drift_x*72)
plt.plot(drift_result.drift_y*72)

plt.show()

In [ ]:
link_r = 0.5 * (np.median(corrected_locs['xc_err']) + np.median(corrected_locs['yc_err']))
linked = link_localisations(corrected_locs, n_frames=n_frames, r_max=link_r, max_dark_time=2)

In [ ]:
oversampling = 1

info = [{
    "Width": width,         # Image width in pixels
    "Height": height,        # Image height in pixels
    "Frames": np.max(loc_data['frame']),      # Total number of frames in the movie
    "Pixelsize": 69,        # pixel size always in nm
}]

image = render.render(
            locs=linked.to_records(index=False),
            info=info,
            blur_method="smooth",
            oversampling=oversampling
        )[1]  # Take the rendered image

In [ ]:
plt.imshow(image, origin='lower', vmax=np.percentile(image, 99))

In [ ]:
oversampling = 1

info = [{
    "Width": width,         # Image width in pixels
    "Height": height,        # Image height in pixels
    "Frames": np.max(loc_data['frame']),      # Total number of frames in the movie
    "Pixelsize": 69,        # pixel size always in nm
}]

image = render.render(
            locs=linked.to_records(index=False),
            info=info,
            blur_method="smooth",
            oversampling=oversampling
        )[1]  # Take the rendered image

region_centers, binary_mask, threshold, metadata = drift_corrector.detect_high_density_regions_from_image(
      smoothed_image=image,
      histogram_bins=1024,
      threshold_percentile=99.94,
      pixelsize=69.0,
      title="Fiducial Detection"
  )

In [ ]:
info = [{
    "Width": width,         # Image width in pixels
    "Height": height,        # Image height in pixels
    "Frames": np.max(linked['frame']),      # Total number of frames in the movie
    "Pixelsize": 69,        # pixel size always in nm
}]

selected_puncta, selection_metadata = drift_corrector.select_puncta_from_regions(
        locs=linked.to_records(index=False),                             # Your localization data
        region_centres=region_centers,         # Output from Step 1
        binary_mask=binary_mask,               # Output from Step 1
        pixelsize=info[0]['Pixelsize'],
        selection_box_size_nm=350.0,         # Box size around each region center (adjust as needed)
        min_localisations_per_region=int(0.075*info[0]['Frames']),      # Minimum locs for valid fiducial (adjust as needed)
        title="Step 2: Puncta Selection",
        create_plot=True
    )

In [ ]:
linked_fremoved = FD.remove_puncta_locs(linked.to_records(index=False), selected_puncta)
linked_fremoved = pd.DataFrame(linked_fremoved)

In [ ]:
IO._write_h5_database(df=linked_fremoved, filepath=localisation_files[0].split('.h5')[0]+'_Undrifted_Linked_FiducialsRemoved.h5', normalise_photons=False)

In [ ]:
linked_fremoved = pd.read_hdf(localisation_files[1])

In [ ]:
oversampling = 8

info = [{
    "Width": width,         # Image width in pixels
    "Height": height,        # Image height in pixels
    "Frames": np.max(linked_fremoved['frame']),      # Total number of frames in the movie
    "Pixelsize": 72,        # pixel size always in nm
}]

image = render.render(
            locs=linked_fremoved.to_records(index=False),
            info=info,
            blur_method="gaussian",
            oversampling=oversampling
        )[1]  # Take the rendered image

In [ ]:
fig, axs = plotter.two_column_plot(height=5, width=5)

axs = plotter.image_plot(axs, image, cmap='hot', colorbar='off')

plt.show()

In [ ]:
# ============================================================================
# Channel Unmixing Test
# ============================================================================

from pyS3M import SM_extractionfunctions

SM_E = SM_extractionfunctions.extract_SMs()

In [ ]:
# Joint spatial-spectral channel unmixing — replaces spectral-only GMM.
# linked_fremoved is already the output of link_localisations (column 'n' present).
# See claude/Cy3B_594_Cells.md for the full rationale.

assigned_locs, metadata = SM_E.unmix_channels_joint_cluster(
    linked_fremoved,
    n_channels=2,
    channels_to_use=['A_R', 'A_G'],
    spatial_cols=['xc', 'yc'],
    spatial_err_cols=['xc_err', 'yc_err'],
    d_threshold=2.0,
    min_cluster_size=3,
    confidence_threshold_isolated=0.90,
    verbose=True,
    plot_results=True,
)

In [ ]:
channel_1 = assigned_locs[assigned_locs['channel'] == 1]
channel_1 = channel_1.reset_index()
channel_0 = assigned_locs[assigned_locs['channel'] == 0]
channel_0 = channel_0.reset_index()

In [ ]:
channel_1

In [ ]:
oversampling = 8
pixel_size = 72

info = [{
    "Width": width,         # Image width in pixels
    "Height": height,        # Image height in pixels
    "Frames": np.max(linked_fremoved['frame']),      # Total number of frames in the movie
    "Pixelsize": pixel_size,        # pixel size always in nm
}]

min_pixel = 4
min_size = min_pixel*(pixel_size/1000)
# Render two channels
_, img1 = render.render(channel_0.to_records(index=False), info, oversampling=oversampling, blur_method='gaussian', min_blur_width=min_size)
_, img2 = render.render(channel_1.to_records(index=False), info, oversampling=oversampling, blur_method='gaussian', min_blur_width=min_size)


In [ ]:
fig, ax = plotter.one_column_plot(height=2, width=2)

ax = plotter.multichannel_overlay_plot(
  ax, [img2, img1],
  cmaps=['hot', 'cyan'],
  pixelsize=(pixel_size/oversampling),
  scalebarsize=10000,
  vmins=[np.percentile(img2, 1), np.percentile(img1, 1)],
  vmaxs=[np.percentile(img2, 99.9), np.percentile(img1, 99.9)],
  brightness_boost=[1, 1],
  scalebarlabel='10 μm'
)
import matplotlib.patches as patches

# xmin = 275
# ymin = 325
# rect = patches.Rectangle(
#     (xmin * oversampling, ymin * oversampling),
#     150 * oversampling,
#     150 * oversampling,
#     linewidth=0.5,
#     edgecolor="white",
#     facecolor="none",
# )

# # Add the patch to the Axes
# ax.add_patch(rect)

# ymin = 105
# xmin = 525
# width = 160
# height = int(width/2.12)
# rect = patches.Rectangle(
#     (ymin * oversampling, xmin * oversampling),
#     width * oversampling,
#     height * oversampling,
#     linewidth=0.5,
#     edgecolor="cyan",
#     facecolor="none",
# )
# ax.add_patch(rect)



folder = '/scratch/sycamore-asap/2026_Multicolour_Paper/For_Talks'
plt.savefig(os.path.join(folder, 'Massive_Cells_Render_Box_Cy3BATTO594.svg'), dpi=600, format='svg')
plt.show()

In [ ]:
# 2.12 ratio

# Create overlay
fig, ax = plotter.one_column_plot(height=2/2.12, width=2)

ymin = 105
xmin = 525
width = 160
height = int(width/2.12)
img1_subset = img1[int(xmin * oversampling):int((xmin+height) * oversampling), int(ymin * oversampling):int((ymin+width) * oversampling)]
img2_subset = img2[int(xmin * oversampling):int((xmin+height) * oversampling), int(ymin * oversampling):int((ymin+width) * oversampling)]


ax = plotter.multichannel_overlay_plot(
  ax, [img2_subset, img1_subset],
  cmaps=['hot', 'cyan'],
  pixelsize=(pixel_size/oversampling),
  scalebarsize=1000,
  vmins=[np.percentile(img2, 1), np.percentile(img1, 1)],
  vmaxs=[np.percentile(img2, 99.9), np.percentile(img1, 99.9)],
  brightness_boost=[1, 1],
  scalebarlabel='1 μm'
)


folder = '/scratch/sycamore-asap/2026_Multicolour_Paper/For_Talks'
plt.savefig(os.path.join(folder, 'Massive_Cells_Render_Zoomin_Mitochondria.svg'), dpi=600, format='svg')
plt.show()

In [ ]:
# Create overlay
fig, ax = plotter.one_column_plot(height=2, width=2)

ymin = 275
xmin = 325
img1_subset = img1[int(xmin * oversampling):int((xmin+150) * oversampling), int(ymin * oversampling):int((ymin+150) * oversampling)]
img2_subset = img2[int(xmin * oversampling):int((xmin+150) * oversampling), int(ymin * oversampling):int((ymin+150) * oversampling)]


ax = plotter.multichannel_overlay_plot(
  ax, [img2_subset, img1_subset],
  cmaps=['hot', 'cyan'],
  pixelsize=(pixel_size/oversampling),
  scalebarsize=1000,
  vmins=[np.percentile(img2, 1), np.percentile(img1, 1)],
  vmaxs=[np.percentile(img2, 99.9), np.percentile(img1, 99.9)],
  brightness_boost=[1, 1],
  scalebarlabel='1 μm'
)

from skimage.draw import line as bresenham_line

r0, c0 = oversampling*4, oversampling*17   # row, col of p0                                                                                                                                       
r1, c1 = oversampling*12, oversampling*20  # row, col of p1
rows_1, cols_1 = bresenham_line(r0, c0, r1, c1)
values_1 = img1_subset[rows_1, cols_1]
# Coordinates of the line we'd like to sample along

ax.plot(cols_1, rows_1, color='white', linewidth=1.5)

r0, c0 = oversampling*(60-25), oversampling*(62+25)    # row, col of p0                                                                                                                                       
r1, c1 = oversampling*(64-25), oversampling*(72+25)  # row, col of p1
rows_2, cols_2 = bresenham_line(r0, c0, r1, c1)
values_2 = img1_subset[rows_2, cols_2]
# Coordinates of the line we'd like to sample along

ax.plot(cols_2, rows_2, color='white', linewidth=1.5, ls='--')


folder = '/scratch/sycamore-asap/2026_Multicolour_Paper/For_Talks'
plt.savefig(os.path.join(folder, 'Massive_Cells_Render_Zoomin.svg'), dpi=600, format='svg')
plt.show()

In [ ]:
def fwhm(sigma):
    return 2*np.sqrt(2*np.log(2))*sigma

In [ ]:
fig, axs = plotter.one_column_plot(npanels=2, ratios=[1,1], height=1.7, width=1)

dr, dc = np.diff(rows_1.astype(float)), np.diff(cols_1.astype(float))
distances_nm = np.concatenate([[0.], np.cumsum(np.sqrt(dr**2 + dc**2))]) * (69.0/oversampling)

bin_size_nm = 69.0 / 4  # e.g. 2 SR pixels per bar — adjust to taste                                                                                                     
                                                                                                                                                                          
bins = np.arange(distances_nm[0], distances_nm[-1] + bin_size_nm, bin_size_nm)
bin_counts, edges = np.histogram(distances_nm, bins=bins, weights=values_1)
bin_norm,   _     = np.histogram(distances_nm, bins=bins)            # pixels per bin
bin_means = np.where(bin_norm > 0, bin_counts / bin_norm, 0.0)       # mean intensity
bin_centres = (edges[:-1] + edges[1:]) / 2


axs[0].bar(bin_centres, bin_means, width=bin_size_nm * 0.75, color='#888', edgecolor='k', linewidth=0.4)
axs[0].set_xlim([0, 600])

from scipy.optimize import curve_fit
from scipy.signal import find_peaks

def gaussian(x, A, mu, sigma):
  return A * np.exp(-(x - mu)**2 / (2 * sigma**2))

def double_gaussian(x, A1, mu1, s1, A2, mu2, s2, offset):
  return gaussian(x, A1, mu1, s1) + gaussian(x, A2, mu2, s2) + offset

# Auto-guess peak positions from the binned data
peaks, props = find_peaks(bin_means, distance=3, prominence=bin_means.max()*0.2)

if len(peaks) >= 2:
  p1, p2 = peaks[np.argsort(props['prominences'])[-2:]]   # two tallest
  p0 = [
      bin_means[p1], bin_centres[p1], bin_size_nm * 2,
      bin_means[p2], bin_centres[p2], bin_size_nm * 2,
      0.0,
  ]
  try:
      popt, pcov = curve_fit(double_gaussian, bin_centres, bin_means, p0=p0,
                             bounds=(0, np.inf), maxfev=5000)
      perr = np.sqrt(np.diag(pcov))

      x_fit = np.linspace(bin_centres[0], bin_centres[-1], 500)
      fwhm_label = fwhm(popt[[2, 5]])
      separation_nm = abs(popt[4] - popt[1])
      axs[0].plot(0, 0, alpha=0, label=f'PS = {separation_nm:.1f} ± 'f'{np.sqrt(perr[1]**2 + perr[4]**2):.1f} nm')
      axs[0].plot(x_fit, gaussian(x_fit, *popt[[0,1,2]]) + popt[6], 'r--', linewidth=0.75, label='FWHM = '+str(int(fwhm_label[0]))+' nm')
      axs[0].plot(x_fit, gaussian(x_fit, *popt[[3,4,5]]) + popt[6], 'g--', linewidth=0.75, label='FWHM = '+str(int(fwhm_label[1]))+' nm')

  except RuntimeError:
      print("Fit did not converge — adjust p0 or bin_size_nm")

#axs[0].legend(loc='best', fontsize=6)
axs[0].set_xlabel('', fontsize=8)
axs[0].set_ylabel('', fontsize=8)
axs[0].set_yticklabels([])
axs[0].set_ylim([1, 30])
axs[0].grid(lw=0.5, alpha=0.5, ls='--', color='gray')

dr, dc = np.diff(rows_2.astype(float)), np.diff(cols_2.astype(float))
distances_nm = np.concatenate([[0.], np.cumsum(np.sqrt(dr**2 + dc**2))]) * (69.0/oversampling)


#########
bins = np.arange(distances_nm[0], distances_nm[-1] + bin_size_nm, bin_size_nm)
bin_counts, edges = np.histogram(distances_nm, bins=bins, weights=values_2)
bin_norm,   _     = np.histogram(distances_nm, bins=bins)            # pixels per bin
bin_means = np.where(bin_norm > 0, bin_counts / bin_norm, 0.0)       # mean intensity
bin_centres = (edges[:-1] + edges[1:]) / 2


axs[1].bar(bin_centres, bin_means, width=bin_size_nm * 0.75, color='#888', edgecolor='k', linewidth=0.4)
axs[1].set_xlim([0, 300])

# Auto-guess peak positions from the binned data
peaks, props = find_peaks(bin_means, distance=3, prominence=bin_means.max()*0.2)

if len(peaks) >= 2:
  p1, p2 = peaks[np.argsort(props['prominences'])[-2:]]   # two tallest
  p0 = [
      bin_means[p1], bin_centres[p1], bin_size_nm * 2,
      bin_means[p2], bin_centres[p2], bin_size_nm * 2,
      0.0,
  ]
  try:
      popt, pcov = curve_fit(double_gaussian, bin_centres, bin_means, p0=p0,
                             bounds=(0, np.inf), maxfev=5000)
      perr = np.sqrt(np.diag(pcov))

      x_fit = np.linspace(bin_centres[0], bin_centres[-1], 500)
      fwhm_label = fwhm(popt[[2, 5]])
      separation_nm = abs(popt[4] - popt[1])
      axs[1].plot(0, 0, alpha=0, label=f'PS = {separation_nm:.1f} ± 'f'{np.sqrt(perr[1]**2 + perr[4]**2):.1f} nm')
      axs[1].plot(x_fit, gaussian(x_fit, *popt[[0,1,2]]) + popt[6], 'r--', linewidth=0.75, label='FWHM = '+str(int(fwhm_label[0]))+' nm')
      axs[1].plot(x_fit, gaussian(x_fit, *popt[[3,4,5]]) + popt[6], 'g--', linewidth=0.75, label='FWHM = '+str(int(fwhm_label[1]))+' nm')

  except RuntimeError:
      print("Fit did not converge — adjust p0 or bin_size_nm")


#axs[1].legend(loc='best', fontsize=6)
axs[1].set_xlabel('distance / nm', fontsize=8)
axs[1].set_ylabel('', fontsize=8)
axs[1].set_yticklabels([])
axs[1].set_ylim([1, 100])
axs[1].grid(lw=0.5, alpha=0.5, ls='--', color='gray')

folder = '/scratch/sycamore-asap/2026_Multicolour_Paper/For_Talks'
plt.savefig(os.path.join(folder, 'Massive_Cells_Cutthroughs.svg'), dpi=600, format='svg')
plt.show()

In [ ]:
from pyS3M.FRCFunctions import fire


In [ ]:
# --- parameters ---
pixel_size_nm = 69      # nm per camera pixel
zoom          = 8      # SR pixels per camera pixel → 6.9 nm SR pixels

# linked_fremoved, width, height already defined in the notebook

# Shift localisations to start from (0,0) within their bounding box                                                                                                                                                                                            
red_xy = channel_0[['xc', 'yc']].to_numpy()                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                             
x_min, y_min = red_xy[:, 0].min(), red_xy[:, 1].min()     
red_xy_shifted = red_xy.copy()                                                                                                                                                                                                                                 
red_xy_shifted[:, 0] -= x_min                                                                                                                                                                                                                                  
red_xy_shifted[:, 1] -= y_min
                                                                                                                                                                                                                                                             
nx_eff_red = int(red_xy[:, 0].max() - x_min) + 1                                                                                                                                                                                                                   
ny_eff_red = int(red_xy[:, 1].max() - y_min) + 1                                                                                                                                                                                                                   
print(f"Effective field: {nx_eff_red} × {ny_eff_red} px  ({nx_eff_red*69:.0f} × {ny_eff_red*69:.0f} nm)")                                                                                                                                                                      
                                                                                                                                                                                                                                                             
resolution_nm_655, frc_mean_655, res_hi_nm_655, res_lo_nm_655 = fire(                                                                                                                                                                                          
  positions     = red_xy_shifted,                                                                                                                                                                                                                            
  nx            = nx_eff_red,                                                                                                                                                                                                                                    
  ny            = ny_eff_red,                                                                                                                                                                                                                                    
  zoom          = zoom,
  n_blocks      = 50,                                                                                                                                                                                                                                        
  pixel_size_nm = pixel_size_nm,                                                                                                                                                                                                                             
)                                                                                                                                                                                                                                                              
print(f"FIRE resolution: {resolution_nm_655:.1f} nm")    
print(f"FIRE bounds: {np.abs(res_hi_nm_655 - res_lo_nm_655):.1f} nm")                                                                                                                                                                                                          

In [ ]:
print(f"FIRE bounds: {np.abs(res_hi_nm_655 - res_lo_nm_655):.1f} nm")                                                                                                                                                                                                          

In [ ]:
# --- parameters ---
pixel_size_nm = 69      # nm per camera pixel
zoom          = 8      # SR pixels per camera pixel → 6.9 nm SR pixels

# linked_fremoved, width, height already defined in the notebook

green_xy = channel_1[['xc', 'yc']].to_numpy()

x_min, y_min = green_xy[:, 0].min(), green_xy[:, 1].min()     
green_xy_shifted = red_xy.copy()                                                                                                                                                                                                                                 
green_xy_shifted[:, 0] -= x_min                                                                                                                                                                                                                                  
green_xy_shifted[:, 1] -= y_min
                                                                                                                                                                                                                                                             
nx_eff_green = int(green_xy[:, 0].max() - x_min) + 1                                                                                                                                                                                                                   
ny_eff_green = int(green_xy[:, 1].max() - y_min) + 1                                                                                                                                                                                                                   
print(f"Effective field: {nx_eff_green} × {ny_eff_green} px  ({nx_eff_green*69:.0f} × {ny_eff_green*69:.0f} nm)")                                                                                                                                                                      
                                                                                                                                                                                                                                                             
resolution_nm_cy3B, frc_mean_cy3B, res_hi_nm_cy3B, res_lo_nm_cy3B = fire(                                                                                                                                                                                          
  positions     = green_xy_shifted,                                                                                                                                                                                                                            
  nx            = nx_eff_green,                                                                                                                                                                                                                                    
  ny            = ny_eff_green,                                                                                                                                                                                                                                    
  zoom          = zoom,
  n_blocks      = 50,                                                                                                                                                                                                                                        
  pixel_size_nm = pixel_size_nm,                                                                                                                                                                                                                             
)                                                                                                                                                                                                                                                              
print(f"FIRE resolution: {resolution_nm_cy3B:.1f} nm")   
print(f"FIRE bounds: {np.abs(res_hi_nm_cy3B - res_lo_nm_cy3B):.1f} nm")                                                                                                                                                                                                          

In [ ]:
print(f"FIRE bounds: {np.abs(res_hi_nm_cy3B - res_lo_nm_cy3B):.1f} nm")  

In [ ]:
fig, ax = plotter.one_column_plot(width=2.3, height=1.2)

sz_red       = max(nx_eff_red, ny_eff_red) * zoom
sz_green      = max(nx_eff_green, ny_eff_green) * zoom

sr_px_nm = pixel_size_nm / zoom        # nm per SR pixel

k_red   = np.arange(len(frc_mean_655))
q_red   = k_red / sz_red                           # cycles per SR pixel
q_nm_red = q_red / sr_px_nm                   # cycles per nm  (= 1 / d_nm)

k_green   = np.arange(len(frc_mean_cy3B))
q_green   = k_green / sz_green                           # cycles per SR pixel
q_nm_green = q_green / sr_px_nm                   # cycles per nm  (= 1 / d_nm)

# keep only below Nyquist
mask_red = (q_red > 0) & (q_red < 0.6)
mask_green = (q_green > 0) & (q_green < 0.6)

# 1/7 crossing in cycles/nm
q_cross_cy3B = 1.0 / resolution_nm_cy3B         # cycles per nm at the FIRE value
q_hi_cy3B    = 1.0 / res_hi_nm_cy3B
q_lo_cy3B    = 1.0 / res_lo_nm_cy3B


# 1/7 crossing in cycles/nm
q_cross_655 = 1.0 / resolution_nm_655         # cycles per nm at the FIRE value
q_hi_655    = 1.0 / res_hi_nm_655
q_lo_655    = 1.0 / res_lo_nm_655

ax = plotter.line_plot(ax, q_nm_red[mask_red], frc_mean_655[mask_red], label='FRC (mean), ATTO 655', color='darkred')
ax = plotter.line_plot(ax, q_nm_green[mask_green], frc_mean_cy3B[mask_green], label='FRC (mean), Cy3B', color='darkgreen')

ax.axhline(1/7, color='gray', ls='--', lw=1, label='1/7 threshold')
ax.axvline(q_cross_655, color='crimson', ls='-', lw=1,
           label=f'FIRE = {resolution_nm_655:.0f} nm')
ax.axvspan(q_lo_655, q_hi_655, alpha=0.15, color='crimson', label='±1σ')

ax.axvline(q_cross_cy3B, color='green', ls='-', lw=1,
           label=f'FIRE = {resolution_nm_cy3B:.0f} nm')
ax.axvspan(q_lo_cy3B, q_hi_cy3B, alpha=0.15, color='green', label='±1σ')

ax.set_xlabel(r'1 / spatial frequency (nm)', fontsize=8)
ax.set_ylabel('FRC', fontsize=8)
ax.set_ylim([0, 1.05])
ax.set_xscale('log')

tick_nm = [500, 100, 20]          # nm values to label
ax.set_xticks([1/d for d in tick_nm])

ax.set_xticklabels([f'{d} nm' for d in tick_nm])
ax.set_xlim(1/500, 1/15)
#ax.legend(fontsize=7)
plt.tight_layout()

folder = '/scratch/sycamore-asap/2026_Multicolour_Paper/For_Talks'
plt.savefig(os.path.join(folder, "FRC_MassiveCells.svg"), dpi=600, format="svg")


In [ ]:
# FRC in linear frequency space (cycles / nm)
fig, ax = plotter.one_column_plot(width=2.3, height=1.2)

ax = plotter.line_plot(ax, q_nm_red[mask_red],   frc_mean_655[mask_red],   label='FRC (mean), ATTO 655', color='darkred')
ax = plotter.line_plot(ax, q_nm_green[mask_green], frc_mean_cy3B[mask_green], label='FRC (mean), Cy3B',     color='darkgreen')

ax.axhline(1/7, color='gray', ls='--', lw=1, label='1/7 threshold')

ax.axvline(q_cross_655, color='crimson', ls='-', lw=1,
           label=f'FIRE = {resolution_nm_655:.0f} nm')
ax.axvspan(q_lo_655, q_hi_655, alpha=0.15, color='crimson', label='±1σ')

ax.axvline(q_cross_cy3B, color='green', ls='-', lw=1,
           label=f'FIRE = {resolution_nm_cy3B:.0f} nm')
ax.axvspan(q_lo_cy3B, q_hi_cy3B, alpha=0.15, color='green', label='±1σ')

ax.set_xlabel(r'spatial frequency / nm$^{-1}$', fontsize=8)
ax.set_ylabel('FRC', fontsize=8)
ax.set_ylim([0, 1.05])
ax.set_xlim(q_nm_red[mask_red][0], q_nm_red[mask_red][-1])

plt.tight_layout()

folder = '/scratch/sycamore-asap/2026_Multicolour_Paper/For_Talks'
plt.savefig(os.path.join(folder, "FRC_MassiveCells_LinearFreq.svg"), dpi=600, format="svg")
